In [1]:
from dx_client import DXClient, DXClientConfig
from dx_client.cache import DuckDBCache

from typing import List
import numpy as np

import dotenv
dotenv.load_dotenv('../../.env')

True

In [2]:
cacheInstance = DuckDBCache()
config = DXClientConfig(
    
)
client =DXClient(config=config, cache=cacheInstance)
client.connect()

In [3]:
dd = client.get_data_dictionary(refresh=False)

In [4]:
# 1. search hypertension related fields


# 在 title、folder_path、description 中搜索
htn = dd[
    dd['title'].str.lower().str.contains('hypertension|blood pressure|systolic|diastolic', na=False) |
    dd['folder_path'].str.lower().str.contains('hypertension|blood.pressure', na=False)
]

htn
# htn[['entity', 'name', 'title', 'type', 'folder_path', 'coding_name']].to_string()


,entity,name,type,primary_key_type,title,description,coding_name,concept,folder_path,is_multi_select,is_sparse_coding,linkout,longitudinal_axis_type,referenced_entity_field,relationship,units
32,participant,p36_i0,string,,Blood pressure device ID | Instance 0,None,,None,Assessment centre/Physical measures/Blood pres...,False,False,http://biobank.ctsu.ox.ac.uk/crystal/field.cgi...,None,,,NaN
33,participant,p36_i1,string,,Blood pressure device ID | Instance 1,None,,None,Assessment centre/Physical measures/Blood pres...,False,False,http://biobank.ctsu.ox.ac.uk/crystal/field.cgi...,None,,,NaN
34,participant,p36_i2,string,,Blood pressure device ID | Instance 2,None,,None,Assessment centre/Physical measures/Blood pres...,False,False,http://biobank.ctsu.ox.ac.uk/crystal/field.cgi...,None,,,NaN
35,participant,p36_i3,string,,Blood pressure device ID | Instance 3,None,,None,Assessment centre/Physical measures/Blood pres...,False,False,http://biobank.ctsu.ox.ac.uk/crystal/field.cgi...,None,,,NaN
36,participant,p37_i0,string,,Blood pressure manual sphygmomanometer device ...,None,,None,Assessment centre/Physical measures/Blood pres...,False,False,http://biobank.ctsu.ox.ac.uk/crystal/field.cgi...,None,,,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32221,participant,p132187,integer,,Source of report of O13 (gestational [pregnanc...,None,data_coding_2171,None,Health-related outcomes/First occurrences/Preg...,False,False,http://biobank.ctsu.ox.ac.uk/crystal/field.cgi...,None,,,NaN
32222,participant,p132188,date,,Date O14 first reported (gestational [pregnanc...,None,data_coding_819,None,Health-related outcomes/First occurrences/Preg...,False,True,http://biobank.ctsu.ox.ac.uk/crystal/field.cgi...,None,,,NaN
32223,participant,p132189,integer,,Source of report of O14 (gestational [pregnanc...,None,data_coding_2171,None,Health-related outcomes/First occurrences/Preg...,False,False,http://biobank.ctsu.ox.ac.uk/crystal/field.cgi...,None,,,NaN
32226,participant,p132192,date,,Date O16 first reported (unspecified maternal ...,None,data_coding_819,None,Health-related outcomes/First occurrences/Preg...,False,True,http://biobank.ctsu.ox.ac.uk/crystal/field.cgi...,None,,,NaN


In [5]:
proteome = dd[dd['entity'].str.contains(r'olink_*')]
proteome_fields = proteome['entity'] + '.' + proteome['name']

proteome_fields 

32803       olink_instance_0.eid
32804      olink_instance_0.a1bg
32805     olink_instance_0.aamdc
32806    olink_instance_0.aarsd1
32807     olink_instance_0.abca2
                  ...           
38650     olink_instance_3.xrcc4
38651      olink_instance_3.yes1
38652    olink_instance_3.ythdf3
38653    olink_instance_3.zbtb16
38654    olink_instance_3.zbtb17
Length: 5852, dtype: str

In [6]:
# cohort = client.create_cohort(name='auto_cohort', entity_fields=['olink_instance_0$A1BG', 'participant$p132601'], filters = {
#     "logic": "and",
#     "pheno_filters": {
#         "logic": "and",
#         "compound": [
#             {
#                 "name": "phenotype",
#                 "logic": "and",
#                 "filters": {
#                     # 字段名 → 条件列表
#                     "participant$p30903_i0": [
#                         {"condition": "between", "values": [0, 2]}
#                     ]
#                 }
#             }
#         ]
#     }
# })

In [7]:
dd[dd['entity'].str.contains('olink')]

,entity,name,type,primary_key_type,title,description,coding_name,concept,folder_path,is_multi_select,is_sparse_coding,linkout,longitudinal_axis_type,referenced_entity_field,relationship,units
32803,olink_instance_0,eid,string,primary_key,Participant ID,None,,None,,False,False,NaN,None,,,NaN
32804,olink_instance_0,a1bg,double,,A1BG;Alpha-1B-glycoprotein,None,,None,,False,False,NaN,None,,,NaN
32805,olink_instance_0,aamdc,double,,AAMDC;Mth938 domain-containing protein,None,,None,,False,False,NaN,None,,,NaN
32806,olink_instance_0,aarsd1,double,,AARSD1;Alanyl-tRNA editing protein Aarsd1,None,,None,,False,False,NaN,None,,,NaN
32807,olink_instance_0,abca2,double,,ABCA2;ATP-binding cassette sub-family A member 2,None,,None,,False,False,NaN,None,,,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38650,olink_instance_3,xrcc4,double,,XRCC4;DNA repair protein XRCC4,None,,None,,False,False,NaN,None,,,NaN
38651,olink_instance_3,yes1,double,,YES1;Tyrosine-protein kinase Yes,None,,None,,False,False,NaN,None,,,NaN
38652,olink_instance_3,ythdf3,double,,YTHDF3;YTH domain-containing family protein 3,None,,None,,False,False,NaN,None,,,NaN
38653,olink_instance_3,zbtb16,double,,ZBTB16;Zinc finger and BTB domain-containing p...,None,,None,,False,False,NaN,None,,,NaN


In [10]:
# 下载完整的olink数据
# 存在的问题：直接将所有column放进来会导致最终得到的df为空，需要分为多块进行整合。然而UKB有提到eid在不同的application数据获取之间会变化，这种切分方法也存在着潜在问题。
# 关于这一点，我需要进行主动测试。eid的变动会造成cohort的合并错误，计划通过比对两次cohort的合并结果是否一致来进行验证
# 更新：正确的方法应该是回避直接的数据下载，最简单的方法是使用cli的extract_table (应该也有对应的API，目前先实现数据提取)，导出数据后通过启动Workstation进行分析（也许可以下载到本地）


olink_cohort_set = dd[dd['entity'].str.contains(r'olink_.*0')]
olink_fields = olink_cohort_set['entity'] + '$' + olink_cohort_set['name']

olink0_cohort = client.create_cohort('olink_cohort_1', entity_fields=olink_fields.tolist()[:500], filters={}, folder="/")

In [11]:
olink0_cohort

DXCohortInfo(id='record-J7GB8ZQJj70b37Bf5z7Bjpvz', name='olink_cohort_1', project='project-J3Y7p9jJj70gKV8XKv2yxqv5', folder='/', state='', description='', created=0, modified=0, participant_count=0, entity_fields=['olink_instance_0$eid', 'olink_instance_0$a1bg', 'olink_instance_0$aamdc', 'olink_instance_0$aarsd1', 'olink_instance_0$abca2', 'olink_instance_0$abhd14b', 'olink_instance_0$abl1', 'olink_instance_0$abo', 'olink_instance_0$abraxas2', 'olink_instance_0$acaa1', 'olink_instance_0$acadm', 'olink_instance_0$acadsb', 'olink_instance_0$acan', 'olink_instance_0$ace', 'olink_instance_0$ace2', 'olink_instance_0$ache', 'olink_instance_0$acot13', 'olink_instance_0$acox1', 'olink_instance_0$acp1', 'olink_instance_0$acp5', 'olink_instance_0$acp6', 'olink_instance_0$acrbp', 'olink_instance_0$acrv1', 'olink_instance_0$acsl1', 'olink_instance_0$acta2', 'olink_instance_0$actn2', 'olink_instance_0$actn4', 'olink_instance_0$acvrl1', 'olink_instance_0$acy1', 'olink_instance_0$acy3', 'olink_insta

In [14]:
dd[dd['title'].str.contains('hypertension')]

,entity,name,type,primary_key_type,title,description,coding_name,concept,folder_path,is_multi_select,is_sparse_coding,linkout,longitudinal_axis_type,referenced_entity_field,relationship,units
20094,participant,p26244,double,,Standard PRS for hypertension (HT),None,,None,Genomics/Polygenic Risk Scores/Standard PRS,False,False,http://biobank.ctsu.ox.ac.uk/crystal/field.cgi...,None,,,relative risk
20095,participant,p26245,double,,Enhanced PRS for hypertension (HT),None,,None,Genomics/Polygenic Risk Scores/Enhanced PRS,False,False,http://biobank.ctsu.ox.ac.uk/crystal/field.cgi...,None,,,relative risk
31324,participant,p131286,date,,Date I10 first reported (essential (primary) h...,None,data_coding_819,None,Health-related outcomes/First occurrences/Circ...,False,True,http://biobank.ctsu.ox.ac.uk/crystal/field.cgi...,None,,,NaN
31325,participant,p131287,integer,,Source of report of I10 (essential (primary) h...,None,data_coding_2171,None,Health-related outcomes/First occurrences/Circ...,False,False,http://biobank.ctsu.ox.ac.uk/crystal/field.cgi...,None,,,NaN
31332,participant,p131294,date,,Date I15 first reported (secondary hypertension),None,data_coding_819,None,Health-related outcomes/First occurrences/Circ...,False,True,http://biobank.ctsu.ox.ac.uk/crystal/field.cgi...,None,,,NaN
31333,participant,p131295,integer,,Source of report of I15 (secondary hypertension),None,data_coding_2171,None,Health-related outcomes/First occurrences/Circ...,False,False,http://biobank.ctsu.ox.ac.uk/crystal/field.cgi...,None,,,NaN
32214,participant,p132180,date,,Date O10 first reported (pre-existing hyperten...,None,data_coding_819,None,Health-related outcomes/First occurrences/Preg...,False,True,http://biobank.ctsu.ox.ac.uk/crystal/field.cgi...,None,,,NaN
32215,participant,p132181,integer,,Source of report of O10 (pre-existing hyperten...,None,data_coding_2171,None,Health-related outcomes/First occurrences/Preg...,False,False,http://biobank.ctsu.ox.ac.uk/crystal/field.cgi...,None,,,NaN
32218,participant,p132184,date,,Date O12 first reported (gestational [pregnanc...,None,data_coding_819,None,Health-related outcomes/First occurrences/Preg...,False,True,http://biobank.ctsu.ox.ac.uk/crystal/field.cgi...,None,,,NaN
32219,participant,p132185,integer,,Source of report of O12 (gestational [pregnanc...,None,data_coding_2171,None,Health-related outcomes/First occurrences/Preg...,False,False,http://biobank.ctsu.ox.ac.uk/crystal/field.cgi...,None,,,NaN
